# 02 — Modeling
Pipeline lengkap scikit-learn:

```
Pipeline
├── struct : StructuralFeatures        (kode wilayah dari building_id + proksi fisika)
├── prep   : ColumnTransformer
│   ├── num : passthrough               (numerik, lokasi, proksi struktur)
│   ├── cat : CategoricalCoder          (kategori konstruksi → pandas category)
│   ├── grp : GroupRelativeFeatures     (bangunan relatif terhadap areanya)
│   ├── te  : HierarchicalTargetEncoder (target encoding berjenjang, out-of-fold)
│   └── nb  : IdNeighborEncoder         (label tetangga ID, out-of-fold)
└── clf    : LGBMClassifier (multiclass)
```
lalu dibungkus `WeightedArgmaxClassifier` (bobot per kelas untuk memaksimalkan macro F1).

Encoder berlabel memakai *cross-fitting* di `fit_transform` sehingga baris train tidak pernah melihat
labelnya sendiri. Karena seluruh pipeline di-fit ulang di setiap fold CV (dan di setiap kandidat tuning),
tidak ada kebocoran target ke data validasi.

**Estimasi waktu di Kaggle CPU:** tuning ±20–30 menit, CV 5-fold ±30–40 menit, model final ±8 menit.

In [ ]:
# === Setup: temukan folder repo (lokal / Kaggle) ===
import sys, subprocess
from pathlib import Path

# Jika dijalankan di Kaggle dan repo belum ada, isi URL GitHub repo kamu (Internet harus ON):
REPO_URL = ""   # contoh: "https://github.com/username/structural-damage-classification.git"

def find_repo_root():
    cands = [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/repo")]
    kin = Path("/kaggle/input")
    if kin.exists():
        cands += [p.parent.parent for p in kin.glob("**/scripts/__init__.py")]
    for c in cands:
        if (c / "scripts" / "__init__.py").exists():
            return c.resolve()
    return None

ROOT = find_repo_root()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    ROOT = Path("/kaggle/working/repo")
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan. Isi REPO_URL atau upload repo sebagai Kaggle Dataset."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import json, time
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.preprocess import load_data, split_xy, output_dir
from scripts import train as T

# ---- konfigurasi ----
TUNE = True        # RandomizedSearchCV pada pipeline (set False untuk memakai parameter default)
N_ITER = 8         # jumlah kandidat random search
TUNE_FRAC = 0.25   # fraksi wilayah (prefix ID 4 digit) untuk tuning
N_SPLITS = 5       # fold cross-validation

OUT = output_dir()
train, test, sample = load_data()
X, y = split_xy(train)
print(X.shape, np.bincount(y)[1:])

## 1. Pipeline

In [ ]:
pipe = T.build_pipeline()
pipe

## 2. Hyperparameter tuning (RandomizedSearchCV pada pipeline)
Tuning dijalankan pada **subsampel wilayah utuh** (bukan baris acak) agar kepadatan tetangga dan
statistik wilayah tetap realistis. Kandidat yang hanya berbeda di parameter LightGBM memakai ulang fitur
yang sudah dihitung (cache `Pipeline(memory=...)`). Parameter default ikut dievaluasi dan dipakai bila
lebih baik.

In [ ]:
best_params = {}
if TUNE:
    t0 = time.time()
    best_params, tuning_results = T.tune_hyperparameters(X, y, n_iter=N_ITER, frac=TUNE_FRAC)
    tuning_results.to_csv(OUT / "reports" / "tuning_results.csv", index=False)
    print(f"tuning selesai dalam {(time.time() - t0) / 60:.1f} menit")
    display(tuning_results.head(10))
print("parameter terpilih (override dari default):", best_params or "default")

## 3. Cross-validation 5-fold (out-of-fold)

In [ ]:
t0 = time.time()
oof, fold_ids, classes = T.cross_validate_oof(X, y, best_params, n_splits=N_SPLITS)
print(f"CV selesai dalam {(time.time() - t0) / 60:.1f} menit")

## 4. Optimasi bobot kelas untuk macro F1
Probabilitas tiap kelas dikalikan bobot `w` (coordinate search). Skor *held-out* dihitung dengan bobot
yang dicari di 4 fold lalu dievaluasi di fold sisanya, sehingga tidak bias optimis.

In [ ]:
cv = T.summarize_cv(oof, y, fold_ids, classes)
print("macro F1 argmax           :", cv["macro_f1_argmax"])
print("macro F1 + bobot (OOF)    :", cv["macro_f1_weighted_oof"])
print("macro F1 held-out per fold:", cv["macro_f1_heldout_per_fold"])
print("macro F1 held-out (rata2) :", cv["macro_f1_heldout_mean"])
print("bobot kelas               :", cv["class_weights"])
pd.DataFrame(cv["report"]).T.round(4)

In [ ]:
pred_oof = classes[(oof * np.asarray(cv["class_weights"])).argmax(1)]
T.plot_confusion(y, pred_oof)
from IPython.display import Image
Image(str(OUT / "reports" / "figures" / "confusion_matrix.png"))

## 5. Model final (seluruh data train) + simpan

In [ ]:
t0 = time.time()
model = T.fit_final_model(X, y, best_params, cv["class_weights"])
model_path = T.save_model(model)
print(f"model final: {(time.time() - t0) / 60:.1f} menit -> {model_path}")

In [ ]:
imp = T.plot_feature_importance(model)
Image(str(OUT / "reports" / "figures" / "feature_importance.png"))

In [ ]:
summary = {"params": {**T.DEFAULT_LGBM_PARAMS, **best_params},
           **{k: v for k, v in cv.items() if k != "report"},
           "per_class_f1": {k: round(v["f1-score"], 4) for k, v in cv["report"].items()
                            if k not in ("accuracy", "macro avg", "weighted avg")},
           "top_features": imp.head(15).round(0).to_dict()}
with open(OUT / "reports" / "cv_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(json.dumps({k: summary[k] for k in ("macro_f1_heldout_mean", "per_class_f1", "class_weights")}, indent=1))

## 6. Catatan interpretasi
* Bandingkan `per_class_f1`: Grade 2 dan 3 biasanya paling sulit karena berada di tengah (tertukar dengan tetangga gradenya).
* Lihat `feature_importance.png`: fitur lokasi (target encoding wilayah, tetangga ID) diharapkan mendominasi,
  diikuti interaksi wilayah × material/fondasi dan proksi struktur.
* Salin angka dari `reports/cv_summary.json` ke bagian **Hasil** di `README.md`.

Lanjut ke `03_submission.ipynb`.